# 记忆评估（Memory Evaluation）

> **构建记忆系统只是成功了一半。验证它是否真的有效，是另一半。记忆评估为你提供指标和工具，回答这些问题：检索能找到正确的记忆吗？智能体会忠实地使用它们吗？系统在提供过时的事实吗？记忆库里潜伏着矛盾吗？**

大多数团队构建记忆系统、部署上线，然后听天由命。当用户抱怨「机器人忘了我说过的话」或「它总是给我过时的信息」时，没有系统化的方法来诊断问题。记忆评估用定量指标和自动评分填补了这个空白。

可以把它想象成给学生批改开卷考试。学生（你的智能体）可以翻笔记（记忆）。你想知道：它找到正确的笔记了吗？它用得准确吗？它选的是最新信息，而不是旧草稿吗？

本 notebook 实现了一个**轻量级评估框架**（evaluation harness，一种测试框架），覆盖四个关键维度：
1. **检索质量：** Recall@K（查全率）、Precision@K（查准率）和平均倒数排名（Mean Reciprocal Rank，MRR）。
2. **忠实度（faithfulness）：** 智能体的回答是否准确反映检索到的记忆，没有幻觉（编造内容）？
3. **时序准确性（temporal accuracy）：** 事实更新之后，检索是否偏好最新版本？
4. **矛盾率（contradiction rate）：** 存储的记忆中有多少比例互相冲突？

**学完本 notebook 你将能够：**
- 构建带真值标注（ground truth）的合成评估数据集。
- 对照已知相关的记忆计算检索指标。
- 使用 LLM-as-judge 模式（让语言模型给质量打分）做忠实度和矛盾评分。
- 运行完整的评估流程并解读结果。


## 关键概念

- **Recall@K（查全率）：** 在所有与查询相关的记忆中，有多大比例出现在前 K 条检索结果里？查全率高，说明系统不会漏掉重要上下文。例如：共有 2 条相关记忆，且都出现在前 5 条结果中，则 recall@5 = 1.0。
- **Precision@K（查准率）：** 在前 K 条检索到的记忆中，有多大比例真正相关？查准率高，说明系统不会让无关记忆污染上下文。例如：检索到的 5 条记忆中有 2 条相关，则 precision@5 = 0.4。
- **平均倒数排名（Mean Reciprocal Rank，MRR）：** 所有查询上，第一条相关记忆的 1/rank 的平均值。它衡量系统多快能把最佳匹配顶到前面。如果第一条相关结果总是排第 1，MRR = 1.0。
- **忠实度（faithfulness）：** 智能体的回答是否准确反映检索到的记忆，没有添加无依据的说法？我们用 LLM-as-judge（让模型给准确性打分）来评估。
- **时序准确性（temporal accuracy）：** 当同一事实存在多个版本时（如旧地址和新地址），检索是否偏好最新的？对跟踪用户状态变化的智能体来说，这一点至关重要。
- **矛盾率（contradiction rate）：** 存储的记忆中有多大比例包含互相冲突的信息？不加处理的矛盾会让智能体的行为前后不一致。
- **LLM-as-judge：** 用语言模型给规则指标衡量不了的主观质量（忠实度、矛盾）打分。研究表明，只要提示词得当，它与人类判断有很好的相关性。


## Architecture

<p align="center">
  <img src="../../images/diagrams/28_memory_evaluation.svg" alt="架构图" width="720"/>
</p>

<details><summary>Mermaid 源码</summary>

```mermaid
flowchart TD
    subgraph Inputs["Evaluation Inputs"]
        MS[("Memory Store\n─────────\nrecords\ntimestamps\nsupersedes")]
        ED["Eval Dataset\n─────────\nqueries\nrelevant_ids\nexpected answers"]
    end

    subgraph Pipeline["Eval Harness"]
        direction TB
        R["Retrieval Metrics\nRecall@K \u00b7 Precision@K \u00b7 MRR"]
        F["Faithfulness Scorer\n(LLM-as-Judge)"]
        T["Temporal Accuracy\nChecker"]
        C["Contradiction\nDetector"]
    end

    subgraph Report["Eval Report"]
        S["Aggregate Scores"]
        D["Per-Query Detail"]
        V["Visualizations"]
    end

    MS --> R & T & C
    ED --> R & F
    R --> S
    F --> S
    T --> S
    C --> S
    S --> D --> V

    style MS fill:#4f46e5,color:#fff
    style ED fill:#059669,color:#fff
    style S fill:#d97706,color:#fff
```

</details>


In [ ]:
# 安装所需依赖包（运行一次）
%pip install -q python-dotenv matplotlib numpy deepagents langchain-openai

我们加载 API 密钥并初始化 Anthropic 客户端。你需要在 `.env` 文件里配置 `LLM_API_KEY`。


In [ ]:
import os
import json
from dataclasses import dataclass, field
from datetime import datetime

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


## 核心实现

我们的评估框架由五个组件组成：

1. **`MemoryRecord`：** 一条带时间戳的记忆条目，带有可选的 `supersedes` 字段用于时序跟踪。（「supersedes」的意思是「这条新记录取代了某条旧记录」。）
2. **`SimpleMemoryStore`：** 一个基于关键词重合度的检索存储（被测系统）。你可以换成自己的存储。
3. **检索指标：** `recall_at_k`、`precision_at_k`、`mrr`，对照真值相关性标签计算。
4. **LLM-as-judge 评分器：** `FaithfulnessJudge` 和 `ContradictionDetector`，用 Claude 做主观评估。
5. **`MemoryEvalHarness`：** 编排所有指标，生成汇总报告。


In [ ]:
@dataclass
class MemoryRecord:
    """一条带评估用元数据的记忆条目。"""

    id: str
    content: str
    timestamp: str  # ISO 8601
    tags: list[str] = field(default_factory=list)
    supersedes: str | None = None  # 本记录所取代的旧记忆的 ID

    def __repr__(self) -> str:
        short = self.content[:50] + "..." if len(self.content) > 50 else self.content
        return f"Memory({self.id}: {short})"

`SimpleMemoryStore` 是我们要评估的检索系统。它用关键词重合度给相关性打分。这是有意为之的简单设计。本 notebook 的重点是评估记忆系统，而不是构建一个花哨的系统。你可以换成自己的存储来测一测。


In [ ]:
class SimpleMemoryStore:
    """一个基于关键词重合度检索的基础记忆存储。

    这是有意为之的简单设计。本 notebook 的重点是
    *评估*记忆系统，而不是构建一个复杂的系统。
    要评估你自己的存储，用你的实现替换这个类即可。
    """

    def __init__(self):
        self.records: dict[str, MemoryRecord] = {}

    def add(self, record: MemoryRecord) -> None:
        """向存储中添加一条记忆记录。"""
        self.records[record.id] = record

    def retrieve(self, query: str, k: int = 5) -> list[MemoryRecord]:
        """按关键词重合度得分检索前 k 条记录。"""
        query_words = set(query.lower().split())
        scored = []
        for record in self.records.values():
            content_words = set(record.content.lower().split())
            overlap = len(query_words & content_words)
            tag_bonus = sum(1 for t in record.tags if t.lower() in query.lower())
            score = overlap + tag_bonus * 2
            if score > 0:
                scored.append((score, record))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [r for _, r in scored[:k]]

    def get(self, record_id: str) -> MemoryRecord | None:
        return self.records.get(record_id)

    def get_all(self) -> list[MemoryRecord]:
        return list(self.records.values())

    def get_temporal_pairs(self) -> list[tuple[MemoryRecord, MemoryRecord]]:
        """找出 new.supersedes 指向旧记录的（旧, 新）记录对。"""
        pairs = []
        for record in self.records.values():
            if record.supersedes and record.supersedes in self.records:
                old = self.records[record.supersedes]
                pairs.append((old, record))
        return pairs

    def __len__(self) -> int:
        return len(self.records)


print("\u2713 MemoryRecord and SimpleMemoryStore defined")

我们构建一个合成数据集：12 条关于用户 Alice 的记忆记录。数据包含基础事实、一对互相矛盾的记忆（喝咖啡的习惯），以及两次时序更新（换工作和搬城市）。`supersedes` 字段标明每条新记录取代了哪条旧记录。


In [ ]:
# ── 构建合成评估数据集 ──
# 场景：个人助理跟踪用户 Alice 的相关事实。

store = SimpleMemoryStore()

memories = [
    # 基础事实
    MemoryRecord("m01", "Alice works as a software engineer at Google", "2024-01-15", ["work", "job"]),
    MemoryRecord("m02", "Alice lives in San Francisco", "2024-01-15", ["location", "home"]),
    MemoryRecord("m03", "Alice prefers Python for scripting tasks", "2024-02-10", ["programming", "preferences"]),
    MemoryRecord("m04", "Alice enjoys hiking and trail running on weekends", "2024-03-01", ["hobbies", "exercise"]),
    MemoryRecord("m05", "Alice is allergic to peanuts and tree nuts", "2024-01-15", ["health", "allergies"]),
    MemoryRecord("m06", "Alice has a golden retriever named Max", "2024-04-01", ["pets"]),
    MemoryRecord("m07", "Alice is training for the SF marathon in October", "2024-05-15", ["exercise", "goals"]),
    MemoryRecord("m08", "Alice started learning Rust for systems programming", "2024-06-01", ["programming"]),
    # 矛盾对：喝咖啡的习惯
    MemoryRecord("m09", "Alice drinks green tea every morning, she switched from coffee", "2024-07-01", ["preferences", "drinks"]),
    MemoryRecord("m10", "Alice loves espresso and drinks three cups daily", "2024-03-15", ["preferences", "drinks"]),
    # 时序更新（取代旧事实）
    MemoryRecord("m11", "Alice joined Anthropic as a senior engineer in March 2025", "2025-03-01", ["work", "job"], supersedes="m01"),
    MemoryRecord("m12", "Alice moved to New York City in January 2025", "2025-01-15", ["location", "home"], supersedes="m02"),
]

for m in memories:
    store.add(m)

print(f"\u2713 Memory store built: {len(store)} records")
print(f"  Temporal pairs: {len(store.get_temporal_pairs())}")
for old, new in store.get_temporal_pairs():
    print(f"    {old.id} \u2192 {new.id}: \"{old.content[:40]}...\" superseded by \"{new.content[:40]}...\"")


# ── 评估用例：带真值相关记忆 ID 的查询 ──

@dataclass
class EvalCase:
    """一条带真值标注的评估查询。"""

    query: str
    relevant_ids: list[str]   # 相关记忆的 ID
    expected_answer: str       # 正确回答应传达的内容
    temporal_preferred: str | None = None  # 应排名最高的 ID（最新的那条）

每个 `EvalCase` 把一个查询与应当相关的记忆 ID 配对。部分用例还标记了 `temporal_preferred` ID：它指向的新记录应当排在过时记录之上。有了这些真值标签，计算检索指标时就不需要猜。


In [ ]:
eval_cases = [
    EvalCase(
        query="Where does Alice work?",
        relevant_ids=["m01", "m11"],
        expected_answer="Alice works at Anthropic as a senior engineer (as of March 2025).",
        temporal_preferred="m11",
    ),
    EvalCase(
        query="Where does Alice live?",
        relevant_ids=["m02", "m12"],
        expected_answer="Alice lives in New York City (moved in January 2025).",
        temporal_preferred="m12",
    ),
    EvalCase(
        query="What programming languages does Alice use?",
        relevant_ids=["m03", "m08"],
        expected_answer="Alice uses Python for scripting and is learning Rust for systems programming.",
    ),
    EvalCase(
        query="What are Alice\'s hobbies and exercise habits?",
        relevant_ids=["m04", "m07"],
        expected_answer="Alice enjoys hiking, trail running, and is training for the SF marathon.",
    ),
    EvalCase(
        query="Does Alice have any food allergies?",
        relevant_ids=["m05"],
        expected_answer="Yes, Alice is allergic to peanuts and tree nuts.",
    ),
    EvalCase(
        query="Does Alice have any pets?",
        relevant_ids=["m06"],
        expected_answer="Yes, Alice has a golden retriever named Max.",
    ),
    EvalCase(
        query="What does Alice like to drink in the morning?",
        relevant_ids=["m09", "m10"],
        expected_answer="Alice drinks green tea every morning. She switched from coffee.",
        temporal_preferred="m09",
    ),
]

print(f"\n\u2713 Eval dataset built: {len(eval_cases)} test queries")
for ec in eval_cases:
    print(f"  \u2022 \"{ec.query}\" \u2192 relevant: {ec.relevant_ids}")

### 检索指标

这些指标把记忆库的检索结果与真值相关性标签做对比。这里不需要调用 LLM，是纯计算。


In [ ]:
# ── 检索指标 ──

def recall_at_k(retrieved_ids: list[str], relevant_ids: list[str], k: int) -> float:
    """在所有相关记忆中，有多大比例出现在前 K 条检索结果里？

    recall@K = |retrieved_top_k \u2229 relevant| / |relevant|
    """
    if not relevant_ids:
        return 1.0
    top_k = set(retrieved_ids[:k])
    relevant = set(relevant_ids)
    return len(top_k & relevant) / len(relevant)


def precision_at_k(retrieved_ids: list[str], relevant_ids: list[str], k: int) -> float:
    """在前 K 条检索到的记忆中，有多大比例真正相关？

    precision@K = |retrieved_top_k \u2229 relevant| / K
    """
    if k == 0:
        return 0.0
    top_k = set(retrieved_ids[:k])
    relevant = set(relevant_ids)
    return len(top_k & relevant) / k


def mean_reciprocal_rank(retrieved_ids: list[str], relevant_ids: list[str]) -> float:
    """第一条相关结果的排名倒数 1/rank。找不到则为 0。"""
    relevant = set(relevant_ids)
    for i, rid in enumerate(retrieved_ids):
        if rid in relevant:
            return 1.0 / (i + 1)
    return 0.0


# ── 快速自检 ──
print("\u2713 Retrieval metric functions defined\n")

example_retrieved = ["m11", "m01", "m03", "m06"]
example_relevant = ["m01", "m11"]

print(f"Example: retrieved={example_retrieved}, relevant={example_relevant}")
print(f"  Recall@3:    {recall_at_k(example_retrieved, example_relevant, k=3):.2f}")
print(f"  Precision@3: {precision_at_k(example_retrieved, example_relevant, k=3):.2f}")
print(f"  MRR:         {mean_reciprocal_rank(example_retrieved, example_relevant):.2f}")

`FaithfulnessJudge` 用 Claude 来打分，判断智能体的回答是否忠实于检索到的记忆。「忠实」指回答中的每个说法都至少有一条源记忆支撑。评分器返回一个 0 到 1 的分数、一句理由，以及无依据说法的列表。


In [ ]:
class FaithfulnessJudge:
    """用 Claude 给回答的忠实度打分，判断其是否忠实于检索到的记忆。

    忠实 = 回答只做出源记忆能够支撑的陈述。
    忠实的回答可以省略信息，但绝不能幻觉出（编造）内容。
    """

    def __init__(self, model):
        self.model = model

    def score(self, query: str, response: str, memories: list[str]) -> dict:
        """对照源记忆给回答的忠实度打分。

        Returns:
            dict，含 'score'（0.0-1.0）、'reasoning' 和 'unsupported_claims'
        """
        memories_text = "\n".join(f"  - {m}" for m in memories)

        result = self.model.invoke([
            {"role": "system", "content": (
                "You are an evaluation judge. Score whether a response is faithful to "
                "the provided source memories. Faithful means: every claim in the response "
                "is supported by at least one source memory. The response may omit info "
                "but must NOT add unsupported claims. Return ONLY a JSON object."
            )},
            {
                "role": "user",
                "content": (
                    f"Query: {query}\n\n"
                    f"Source memories:\n{memories_text}\n\n"
                    f"Response to evaluate: {response}\n\n"
                    "Return a JSON object with:\n"
                    '- "score": float 0.0-1.0 (1.0 = fully faithful)\n'
                    '- "reasoning": one sentence explaining the score\n'
                    '- "unsupported_claims": list of claims not supported by source memories\n'
                ),
            },
        ])

        try:
            text = to_text(result.content).strip()
            if text.startswith("```"):
                text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
            return json.loads(text)
        except (json.JSONDecodeError, IndexError):
            return {"score": 0.5, "reasoning": "Could not parse judge response", "unsupported_claims": []}


print("\u2713 FaithfulnessJudge defined (uses Claude as evaluator)")

`temporal_accuracy_score` 检查检索是否偏好较新的事实，而不是被取代的旧事实。对每个带 `temporal_preferred` ID 的评估用例，它会验证新记录在结果中的排名高于旧记录。


In [ ]:
def temporal_accuracy_score(
    store: SimpleMemoryStore,
    eval_cases: list[EvalCase],
    k: int = 5,
) -> dict:
    """检查检索是否偏好较新的事实而非被取代的旧事实。

    对每个带 temporal_preferred ID 的评估用例，检查该 ID
    的排名是否高于被取代的旧记忆。

    Returns:
        dict，含 'score'（正确比例）和逐用例的 'details'
    """
    temporal_cases = [ec for ec in eval_cases if ec.temporal_preferred]
    if not temporal_cases:
        return {"score": 1.0, "details": [], "n_cases": 0}

    details = []
    correct = 0
    for ec in temporal_cases:
        retrieved = store.retrieve(ec.query, k=k)
        retrieved_ids = [r.id for r in retrieved]
        preferred = ec.temporal_preferred

        if preferred in retrieved_ids:
            preferred_rank = retrieved_ids.index(preferred)
            is_correct = True
            for rid in ec.relevant_ids:
                if rid != preferred and rid in retrieved_ids:
                    if retrieved_ids.index(rid) < preferred_rank:
                        is_correct = False
                        break
            if is_correct:
                correct += 1
        else:
            is_correct = False

        details.append({
            "query": ec.query,
            "preferred": preferred,
            "retrieved_order": retrieved_ids,
            "correct": is_correct,
        })

    return {"score": correct / len(temporal_cases), "details": details, "n_cases": len(temporal_cases)}

`ContradictionDetector` 用 Claude 在存储的记忆中查找互相冲突的说法。它把记忆分批处理来控制 API 成本，然后让模型找出对同一主体做出不兼容陈述的记忆对。矛盾率是所有可能的记忆对中发生冲突的比例。


In [ ]:
class ContradictionDetector:
    """用 Claude 检测记忆之间的矛盾。"""

    def __init__(self, model):
        self.model = model

`scan` 方法把记忆分批交给 Claude，让它找出互相矛盾的记忆对。它按组处理记忆以压低 API 成本。每条矛盾都包含两条冲突记忆的 ID 和一段解释。


In [ ]:
def scan(self, memories: list[MemoryRecord], batch_size: int = 15) -> dict:
    """用 LLM-as-judge 扫描记忆条目中的矛盾。

    将记忆分批处理以控制成本，而不是检查全部 O(n\u00b2) 个记忆对。

    Returns:
        dict，含 'contradiction_rate'、'contradictions' 列表和 'n_found'
    """
    contents = [f"[{m.id}] ({m.timestamp}) {m.content}" for m in memories]

    # 分批处理
    batches = [contents[i:i + batch_size] for i in range(0, len(contents), batch_size)]

    all_contradictions = []
    for batch in batches:
        memories_text = "\n".join(batch)

        result = self.model.invoke([
            {"role": "system", "content": (
                "You are a contradiction detector. Given a list of memory entries, "
                "identify any pairs that contradict each other. Two memories contradict "
                "if they make incompatible claims about the same subject. "
                "A newer fact superseding an older one IS a contradiction to flag. "
                "Return ONLY a JSON object."
            )},
            {
                "role": "user",
                "content": (
                    f"Memory entries:\n{memories_text}\n\n"
                    "Return a JSON object with:\n"
                    '- "contradictions": list of objects each with "id_a", "id_b", "explanation"\n'
                    '- "count": number of contradictions found\n'
                    "Use empty list if no contradictions."
                ),
            },
        ])

        try:
            text = to_text(result.content).strip()
            if text.startswith("```"):
                text = text.split("\n", 1)[1].rsplit("```", 1)[0].strip()
            parsed = json.loads(text)
            all_contradictions.extend(parsed.get("contradictions", []))
        except (json.JSONDecodeError, IndexError):
            pass

    n_pairs = len(memories) * (len(memories) - 1) // 2
    return {
        "contradiction_rate": len(all_contradictions) / max(n_pairs, 1),
        "contradictions": all_contradictions,
        "n_found": len(all_contradictions),
        "n_memories": len(memories),
    }

ContradictionDetector.scan = scan

print("\u2713 temporal_accuracy_score and ContradictionDetector defined")

`MemoryEvalHarness` 把全部四个评估维度编排成一份报告。它把检索指标、忠实度评分器、时序检查器和矛盾检测器串在一起。构造函数还会创建一个 LLM 客户端，用于根据检索到的记忆生成测试回答。


In [ ]:
class MemoryEvalHarness:
    """把所有记忆评估指标编排成一份统一的报告。

    Usage:
        harness = MemoryEvalHarness(store, eval_cases)
        report = harness.run(k=3)
    """

    def __init__(
        self,
        store: SimpleMemoryStore,
        eval_cases: list[EvalCase],
        model,
    ):
        self.store = store
        self.eval_cases = eval_cases
        self.faithfulness_judge = FaithfulnessJudge(model=model)
        self.contradiction_detector = ContradictionDetector(model=model)
        self.model = model

    def _generate_response(self, query: str, memories: list[MemoryRecord]) -> str:
        """根据检索到的记忆生成智能体回答。"""
        context = "\n".join(f"- {m.content}" for m in memories)
        result = self.model.invoke([
            {"role": "system", "content": (
                "Answer the user's question using ONLY the provided memories. Be concise."
            )},
            {"role": "user", "content": f"Memories:\n{context}\n\nQuestion: {query}"},
        ])
        return to_text(result.content)


`run` 方法执行完整的评估流程。它为每个查询检索记忆，计算检索指标，生成回答并评估忠实度，检查时序准确性，并扫描矛盾。它返回一份结构化报告，含汇总分数和逐查询明细。


`_eval_retrieval` 为每个评估用例运行检索指标（Recall@K、Precision@K、MRR）。它从存储中检索记忆，把结果 ID 与真值标签对比。这一步不调用 LLM。


In [ ]:
def _eval_retrieval(self, k: int, verbose: bool) -> tuple[list, list, list, list]:
    """为所有评估用例计算检索指标。

    Returns:
        元组 (all_recall, all_precision, all_mrr, retrieval_details)。
    """
    if verbose:
        print("\n\u2500\u2500 Retrieval Metrics \u2500\u2500")

    retrieval_details = []
    all_recall, all_precision, all_mrr = [], [], []

    for ec in self.eval_cases:
        retrieved = self.store.retrieve(ec.query, k=k)
        retrieved_ids = [r.id for r in retrieved]

        r = recall_at_k(retrieved_ids, ec.relevant_ids, k)
        p = precision_at_k(retrieved_ids, ec.relevant_ids, k)
        m = mean_reciprocal_rank(retrieved_ids, ec.relevant_ids)

        all_recall.append(r)
        all_precision.append(p)
        all_mrr.append(m)

        retrieval_details.append({
            "query": ec.query, "retrieved_ids": retrieved_ids,
            "relevant_ids": ec.relevant_ids,
            "recall_at_k": r, "precision_at_k": p, "mrr": m,
        })

        if verbose:
            status = "\u2713" if r == 1.0 else "\u2717"
            print(f"  {status} \"{ec.query}\"")
            print(f"    Retrieved: {retrieved_ids} | Relevant: {ec.relevant_ids}")
            print(f"    Recall@{k}={r:.2f}  Precision@{k}={p:.2f}  MRR={m:.2f}")

    return all_recall, all_precision, all_mrr, retrieval_details

MemoryEvalHarness._eval_retrieval = _eval_retrieval


`_eval_faithfulness` 根据检索到的记忆生成回答，然后让 Claude 判断回答是否忠实。它独立评估每个查询，并收集所有无依据的说法。


In [ ]:
def _eval_faithfulness(self, k: int, verbose: bool) -> tuple[list, list]:
    """用 LLM-as-judge 为所有评估用例评估忠实度。

    Returns:
        元组 (all_faith_scores, faithfulness_details)。
    """
    if verbose:
        print("\n\u2500\u2500 Faithfulness (LLM-as-Judge) \u2500\u2500")

    faithfulness_details = []
    all_faith = []

    for ec in self.eval_cases:
        retrieved = self.store.retrieve(ec.query, k=k)
        memory_texts = [r.content for r in retrieved]
        response = self._generate_response(ec.query, retrieved)
        faith_result = self.faithfulness_judge.score(ec.query, response, memory_texts)
        faith_score = faith_result.get("score", 0.5)
        all_faith.append(faith_score)

        faithfulness_details.append({
            "query": ec.query, "response": response,
            "score": faith_score,
            "reasoning": faith_result.get("reasoning", ""),
            "unsupported": faith_result.get("unsupported_claims", []),
        })

        if verbose:
            status = "\u2713" if faith_score >= 0.8 else "\u2717"
            print(f"  {status} \"{ec.query}\" \u2192 faithfulness={faith_score:.2f}")
            for claim in faith_result.get("unsupported_claims", []):
                print(f"    \u26a0 Unsupported: {claim}")

    return all_faith, faithfulness_details

MemoryEvalHarness._eval_faithfulness = _eval_faithfulness


`run` 方法编排整个流程。它调用上面的检索与忠实度辅助方法，然后运行时序准确性检查和矛盾扫描，最后返回包含所有分数的结构化报告。


`run` 方法现在是一个轻量的编排器。它调用检索指标、忠实度评分、时序准确性和矛盾检测的辅助方法，然后组装报告。


In [ ]:
def run(self, k: int = 3, verbose: bool = True) -> dict:
    """运行完整的评估流程。

    Args:
        k: 每个查询检索的记忆条数。
        verbose: 评估过程中是否打印进度。

    Returns:
        dict，含汇总分数和逐查询明细。
    """
    if verbose:
        print("=" * 60)
        print("MEMORY EVALUATION REPORT")
        print("=" * 60)

    # 1. Retrieval metrics
    all_recall, all_precision, all_mrr, retrieval_details = self._eval_retrieval(k, verbose)

    # 2. Faithfulness
    all_faith, faithfulness_details = self._eval_faithfulness(k, verbose)

    # 3-4. Temporal accuracy and contradiction scan
    temporal, contradictions = self._eval_temporal_and_contradictions(k, verbose)

    return self._build_report(
        k, verbose, all_recall, all_precision, all_mrr,
        retrieval_details, all_faith, faithfulness_details,
        temporal, contradictions,
    )

MemoryEvalHarness.run = run


`_eval_temporal_and_contradictions` 检查两件事。第一，检索是否偏好较新的事实而非被取代的旧事实。第二，存储的记忆之间有没有互相冲突。


In [ ]:
def _eval_temporal_and_contradictions(self, k: int, verbose: bool) -> tuple[dict, dict]:
    """运行时序准确性检查和矛盾扫描。"""
    if verbose:
        print("\n\u2500\u2500 Temporal Accuracy \u2500\u2500")
    temporal = temporal_accuracy_score(self.store, self.eval_cases, k=k)
    if verbose:
        for d in temporal["details"]:
            status = "\u2713" if d["correct"] else "\u2717"
            print(f"  {status} \"{d['query']}\" \u2192 preferred={d['preferred']}, retrieved={d['retrieved_order']}")

    if verbose:
        print("\n\u2500\u2500 Contradiction Scan \u2500\u2500")
    contradictions = self.contradiction_detector.scan(self.store.get_all())
    if verbose:
        print(f"  Found {contradictions['n_found']} contradiction(s) in {contradictions['n_memories']} memories")
        for c in contradictions["contradictions"]:
            print(f"    \u26a0 {c['id_a']} \u2194 {c['id_b']}: {c.get('explanation', '')}")

    return temporal, contradictions

MemoryEvalHarness._eval_temporal_and_contradictions = _eval_temporal_and_contradictions


`_build_report` 把所有分数汇总进一个结构化字典，并打印汇总表。这是评估流程的最后一步。


In [ ]:
def _build_report(
    self, k, verbose, all_recall, all_precision, all_mrr,
    retrieval_details, all_faith, faithfulness_details,
    temporal, contradictions,
) -> dict:
    """组装最终评估报告并打印汇总。"""
    report = {
        "retrieval": {
            "mean_recall_at_k": float(np.mean(all_recall)),
            "mean_precision_at_k": float(np.mean(all_precision)),
            "mean_mrr": float(np.mean(all_mrr)),
            "k": k,
            "details": retrieval_details,
        },
        "faithfulness": {
            "mean_score": float(np.mean(all_faith)),
            "details": faithfulness_details,
        },
        "temporal": temporal,
        "contradictions": contradictions,
    }

    if verbose:
        print("\n" + "=" * 60)
        print("SUMMARY")
        print("=" * 60)
        print(f"  Retrieval Recall@{k}:    {report['retrieval']['mean_recall_at_k']:.2f}")
        print(f"  Retrieval Precision@{k}: {report['retrieval']['mean_precision_at_k']:.2f}")
        print(f"  Mean Reciprocal Rank:    {report['retrieval']['mean_mrr']:.2f}")
        print(f"  Faithfulness:            {report['faithfulness']['mean_score']:.2f}")
        print(f"  Temporal Accuracy:       {report['temporal']['score']:.2f}")
        print(f"  Contradiction Rate:      {report['contradictions']['contradiction_rate']:.2%}")

    return report

MemoryEvalHarness._build_report = _build_report


## 使用示例：完整评估流程

让我们对合成数据集运行评估框架。框架会：

1. 为每个查询检索记忆，计算 Recall@K、Precision@K 和 MRR。
2. 根据检索到的记忆生成回答，并通过 Claude 为其忠实度评分。
3. 检查时序更新是否排在被取代的旧事实之上。
4. 扫描整个存储，找出矛盾。


In [ ]:
harness = MemoryEvalHarness(store, eval_cases)
report = harness.run(k=3)

我们把结果画成两个面板。左图是每个指标的汇总分数。右图逐查询拆解查全率和忠实度。这样一眼就能看出记忆系统擅长哪些查询、在哪些查询上吃力。


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 面板 1：各指标汇总分数 ──
metrics = [
    ("Recall@K", report["retrieval"]["mean_recall_at_k"]),
    ("Precision@K", report["retrieval"]["mean_precision_at_k"]),
    ("MRR", report["retrieval"]["mean_mrr"]),
    ("Faithfulness", report["faithfulness"]["mean_score"]),
    ("Temporal\nAccuracy", report["temporal"]["score"]),
]
names = [m[0] for m in metrics]
values = [m[1] for m in metrics]
colors = ["#4f46e5", "#4f46e5", "#4f46e5", "#059669", "#d97706"]

bars = axes[0].bar(names, values, color=colors, width=0.6, alpha=0.9)
axes[0].set_ylim(0, 1.15)
axes[0].set_ylabel("Score", fontsize=12)
axes[0].set_title("Memory Evaluation Metrics", fontsize=13, fontweight="bold")
axes[0].axhline(y=1.0, color="gray", linestyle="--", alpha=0.3)
for bar, val in zip(bars, values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
        f"{val:.2f}", ha="center", fontweight="bold", fontsize=11,
    )

# ── 面板 2：逐查询拆解 ──
queries = [d["query"][:25] + "..." if len(d["query"]) > 25 else d["query"]
           for d in report["retrieval"]["details"]]
recall_vals = [d["recall_at_k"] for d in report["retrieval"]["details"]]
faith_vals = [d["score"] for d in report["faithfulness"]["details"]]

x = np.arange(len(queries))
width = 0.3
axes[1].barh(x - width / 2, recall_vals, width, label="Recall@K", color="#4f46e5", alpha=0.85)
axes[1].barh(x + width / 2, faith_vals, width, label="Faithfulness", color="#059669", alpha=0.85)
axes[1].set_yticks(x)
axes[1].set_yticklabels(queries, fontsize=8)
axes[1].set_xlim(0, 1.15)
axes[1].set_xlabel("Score", fontsize=11)
axes[1].set_title("Per-Query Breakdown", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=9, loc="lower right")
axes[1].axvline(x=1.0, color="gray", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

# 打印矛盾汇总
if report["contradictions"]["contradictions"]:
    print(f"\n\u26a0 {report['contradictions']['n_found']} contradiction(s) detected:")
    for c in report["contradictions"]["contradictions"]:
        print(f"  {c['id_a']} \u2194 {c['id_b']}: {c.get('explanation', '')}")
else:
    print("\n\u2713 No contradictions detected")

我们来细看逐查询的忠实度分数。对每个查询，可以看到智能体生成的回答、评分器的理由，以及所有无依据的说法。这些细节帮你定位忠实度是在哪里出问题的。


In [ ]:
# ── 查看单条忠实度结果 ──

print("=== Per-Query Faithfulness Detail ===\n")
for detail in report["faithfulness"]["details"]:
    score = detail["score"]
    icon = "\u2713" if score >= 0.8 else "\u2717"
    print(f"{icon} [{score:.2f}] {detail['query']}")
    print(f"  Response: {detail['response'][:120]}...")
    print(f"  Reasoning: {detail['reasoning']}")
    if detail["unsupported"]:
        for claim in detail["unsupported"]:
            print(f"  \u26a0 Unsupported: {claim}")
    print()

## 讨论与权衡

### 优点
- **系统化胜过轶事。** 定量指标用可度量的分数取代了「看起来能用」。在迭代记忆系统时，回归测试成为可能。
- **指标分解。** 把检索质量、忠实度、时序准确性和矛盾分开衡量，你能定位系统*在哪里*失败，而不只知道*它失败了*。
- **LLM-as-judge 可扩展。** 自动化的忠实度和矛盾评分可以按任意规模运行，这是人工审阅做不到的。研究表明，只要提示词得当，LLM 评分与人类评分有很好的相关性。
- **评估数据集可复用。** 一旦构建完成，评估数据集就能作为回归测试，跨越记忆系统版本、检索算法变更和模型升级反复使用。

### 缺点
- **真值标签成本高。** 构建带已知相关记忆 ID 的评估数据集需要人工标注。合成数据集（比如我们的）可能无法反映真实世界的分布。
- **LLM-as-judge 有偏差。** 评分模型可能过于宽松、前后不一致，或与生成模型有同样的盲区。跨模型评分和校准有帮助，但无法彻底消除。
- **关键词检索的局限。** 我们的 `SimpleMemoryStore` 用关键词重合度，会漏掉语义匹配。生产系统应针对基于嵌入或混合检索来评估。
- **矛盾检测有噪声。** LLM 可能把正常更新（如换工作）标记为矛盾，也可能漏掉细微的逻辑冲突。需要调整阈值。
- **没有下游任务评估。** 我们孤立地衡量检索和忠实度。完整的评估还应衡量端到端的任务成败：更好的记忆是否真的带来更好的智能体表现？

### 扩展评估框架

| 扩展方向 | 说明 |
|-----------|-------------|
| **基于嵌入的检索** | 用向量相似度取代关键词重合度，让评估更贴近真实 |
| **人工评估协议** | 在 LLM-as-judge 之外加入人工评分，并衡量标注者之间的一致性 |
| **覆盖率指标** | 给定一段对话，有多少比例可提取的事实最终进入了记忆库？ |
| **延迟跟踪** | 在质量指标之外，度量检索和生成的延迟（响应时间） |
| **A/B 对比** | 在同一评估集上运行两种记忆配置，并用统计检验做比较 |
| **陈旧度评分** | 在时序准确性之外，度量智能体在回答中使用过时信息的频率 |


## 延伸阅读

- [RAGAS：面向 RAG 流程的评估框架](https://docs.ragas.io?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 检索增强生成（RAG）的行业标准指标
- [Maharana 等人，「Evaluating Very Long-Term Conversational Memory of LLM Agents」（2024）](https://arxiv.org/abs/2402.17753) - 长期对话记忆的 LoCoMo 基准测试
- [Wu 等人，「LongMemEval: Benchmarking Chat Assistants on Long-Term Interactive Memory」（2024）](https://arxiv.org/abs/2410.10813) - 多会话记忆评估基准
- [Zheng 等人，「Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena」（2023）](https://arxiv.org/abs/2306.05685) - 用 LLM 做自动评估的奠基性研究
- [Anthropic：构建对话式 AI](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - 基于 Claude 的多轮对话模式
- [LangSmith 评估](https://docs.smith.langchain.com?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) - LLM 应用的生产级评估工具

---

*← 上一篇：[27 - Zep 记忆](../27_zep_memory/) · 下一篇：[29 - 记忆基准测试（LoCoMo）](../29_memory_benchmarks_LoCoMo/) →*


## 🧪 动手练习

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：自定义领域测试集
为你关心的一个领域（如菜谱、项目管理、旅行规划）编写 10 组问答对。创建 `MemoryRecord` 条目作为真值。把它们连同 `SimpleMemoryStore` 一起跑 `MemoryEvalHarness`，然后查看每道题的分数。

### 挑战 2：指标相关性分析
在 20 个问题上运行评估框架，收集所有指标分数（Recall@K、Precision@K、MRR、token F1、ROUGE-L、`FaithfulnessJudge` 分数）。计算指标两两之间的 Pearson 相关系数。找出哪些自动指标最能预测 LLM 评分器的分数。

### 挑战 3：系统正面对比
实现第二个检索后端（例如 06 向量库记忆中基于嵌入的搜索），接口与 `SimpleMemoryStore` 相同。让两个后端跑同一个 `MemoryEvalHarness` 测试集。比较它们在所有指标上的得分，讨论哪种检索方式在哪类问题上占优。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--28-memory-evaluation--memory-evaluation)
